# 📊 F1 Raw Data Profiling
**Mục tiêu:** Phân tích dữ liệu thô SAU khi crawl, TRƯỚC khi clean — để đưa ra chiến lược cleaning đúng.

### 7 Kỹ thuật áp dụng:
1. Data Consolidation & Unification
2. Data Profiling (Descriptive Stats)
3. Data Sanitization (Invalid values, whitespace)
4. Low Variance Filtering (99% threshold)
5. Missing Value Analysis & Visualization
6. Deduplication Check
7. Datetime Feature Detection


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='darkgrid', palette='muted')

RAW_DIR = Path('../data/raw')
LOW_VARIANCE_THRESHOLD = 0.99   # Cột có >99% cùng giá trị → loại
HIGH_MISSING_THRESHOLD = 0.20   # Cột có >20% null → cân nhắc drop
print('Setup OK ✅')

---
## 1️⃣ Data Consolidation & Unification
Gộp tất cả session-based files của từng endpoint, kiểm tra Type Matching và Schema Alignment.

In [ ]:
def load_endpoint(endpoint: str) -> pd.DataFrame:
    """Load và gộp tất cả file CSV/Parquet của một endpoint."""
    ep_dir = RAW_DIR / endpoint
    if not ep_dir.exists():
        return pd.DataFrame()
    files = list(ep_dir.glob('**/*.csv')) + list(ep_dir.glob('**/*.parquet'))
    if not files:
        return pd.DataFrame()

    frames = []
    schema_issues = []
    ref_dtypes = None
    ref_cols = None

    for f in files:
        df = pd.read_parquet(f) if f.suffix == '.parquet' else pd.read_csv(f, low_memory=False)
        if df.empty:
            continue

        # --- Kỹ thuật 1a: Schema Alignment ---
        if ref_cols is None:
            ref_cols = set(df.columns)
            ref_dtypes = df.dtypes
        else:
            extra = set(df.columns) - ref_cols
            missing = ref_cols - set(df.columns)
            if extra or missing:
                schema_issues.append({'file': f.name, 'extra_cols': list(extra), 'missing_cols': list(missing)})

            # --- Kỹ thuật 1b: Type Matching ---
            for col in ref_cols & set(df.columns):
                if ref_dtypes[col] != df[col].dtype:
                    schema_issues.append({'file': f.name, 'col': col,
                                          'expected': str(ref_dtypes[col]), 'got': str(df[col].dtype)})
        frames.append(df)

    if schema_issues:
        print(f'⚠️  Schema/Type issues in [{endpoint}]:')
        display(pd.DataFrame(schema_issues))

    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    combined['_source_endpoint'] = endpoint
    return combined


# Load tất cả endpoints tìm thấy trong data/raw/
ENDPOINTS = [d.name for d in RAW_DIR.iterdir() if d.is_dir()]
print(f'Found endpoints: {ENDPOINTS}')

dfs = {}
for ep in ENDPOINTS:
    df = load_endpoint(ep)
    if not df.empty:
        dfs[ep] = df
        print(f'  ✅ {ep}: {len(df):,} rows × {len(df.columns)} cols')
    else:
        print(f'  ⚫ {ep}: no data')

In [ ]:
# Biểu đồ: số lượng dòng theo endpoint
sizes = {ep: len(df) for ep, df in dfs.items()}
fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.barh(list(sizes.keys()), list(sizes.values()), color=sns.color_palette('muted', len(sizes)))
ax.bar_label(bars, labels=[f'{v:,}' for v in sizes.values()], padding=4, fontsize=9)
ax.set_xlabel('Number of rows')
ax.set_title('📦 Row count per endpoint (raw data)')
plt.tight_layout()
plt.show()

---
## 2️⃣ Data Profiling — Descriptive Statistics & Unique Values
Chọn endpoint cần phân tích chi tiết bên dưới.

In [ ]:
ENDPOINT_TO_PROFILE = 'laps'   # ← Đổi tên endpoint ở đây

if ENDPOINT_TO_PROFILE not in dfs:
    print(f'❌ Endpoint "{ENDPOINT_TO_PROFILE}" not loaded. Available: {list(dfs.keys())}')
else:
    df = dfs[ENDPOINT_TO_PROFILE].copy()

    print(f'=== [{ENDPOINT_TO_PROFILE}] Shape: {df.shape} ===')

    # Descriptive statistics
    print('\n--- Descriptive Statistics ---')
    display(df.describe(include='all').T.style.background_gradient(cmap='Blues', subset=['count']))

    # Unique value analysis
    print('\n--- Unique Value Analysis ---')
    unique_info = pd.DataFrame({
        'dtype': df.dtypes,
        'nunique': df.nunique(),
        'null_count': df.isnull().sum(),
        'null_%': (df.isnull().sum() / len(df) * 100).round(2),
        'sample_values': [df[c].dropna().unique()[:5].tolist() for c in df.columns]
    })
    unique_info['likely_type'] = unique_info.apply(
        lambda r: 'ID/Key' if r.nunique == len(df) else
                  'Categorical' if r.nunique < 30 else
                  'High-card text' if r.dtype == object else 'Numeric', axis=1
    )
    display(unique_info.style.background_gradient(cmap='Oranges', subset=['null_%']))

---
## 3️⃣ Data Sanitization
### 3a. Giá trị phi lý (giá trị âm trong cột số)
### 3b. Whitespace detection

In [ ]:
if ENDPOINT_TO_PROFILE in dfs:
    df = dfs[ENDPOINT_TO_PROFILE].copy()
    numeric_cols = df.select_dtypes(include='number').columns

    # 3a. Negative values
    neg_report = []
    for col in numeric_cols:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            neg_report.append({'column': col, 'negative_count': n_neg,
                                'pct': round(n_neg / len(df) * 100, 2),
                                'min_value': df[col].min()})

    if neg_report:
        print('⚠️  Negative values found:')
        display(pd.DataFrame(neg_report))
    else:
        print('✅ No negative values in numeric columns')

    # 3b. Whitespace-only strings
    ws_report = []
    for col in df.select_dtypes(include='object').columns:
        n_ws = df[col].astype(str).str.strip().eq('').sum()
        if n_ws > 0:
            ws_report.append({'column': col, 'blank_count': n_ws,
                               'pct': round(n_ws / len(df) * 100, 2)})

    if ws_report:
        print('\n⚠️  Whitespace/blank strings found:')
        display(pd.DataFrame(ws_report))
    else:
        print('✅ No whitespace-only strings')

---
## 4️⃣ Low Variance Filtering (≥99% same value → useless column)

In [ ]:
if ENDPOINT_TO_PROFILE in dfs:
    df = dfs[ENDPOINT_TO_PROFILE].copy()

    lv_report = []
    for col in df.columns:
        if col == '_source_endpoint':
            continue
        top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]
        top_val = df[col].value_counts(dropna=False).index[0]
        if top_freq >= LOW_VARIANCE_THRESHOLD:
            lv_report.append({'column': col, 'dominant_value': top_val,
                               'dominance_%': round(top_freq * 100, 2)})

    if lv_report:
        print(f'⚠️  Low-variance columns (≥{LOW_VARIANCE_THRESHOLD*100:.0f}% same value) → candidate for DROP:')
        display(pd.DataFrame(lv_report).style.background_gradient(cmap='Reds', subset=['dominance_%']))
    else:
        print('✅ No low-variance columns found')

---
## 5️⃣ Missing Value Analysis & Visualization

In [ ]:
def plot_missing(df: pd.DataFrame, title: str = '', threshold: float = 0.0):
    miss = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    miss = miss[miss > threshold]
    if miss.empty:
        print(f'✅ No missing values above {threshold}% threshold')
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, len(miss) * 0.35 + 1)))

    # Bar chart
    colors = ['#e74c3c' if v > HIGH_MISSING_THRESHOLD*100 else '#f39c12' if v > 5 else '#2ecc71'
              for v in miss.values]
    axes[0].barh(miss.index, miss.values, color=colors)
    axes[0].axvline(HIGH_MISSING_THRESHOLD * 100, color='red', linestyle='--', label=f'Drop threshold ({HIGH_MISSING_THRESHOLD*100:.0f}%)')
    axes[0].set_xlabel('Missing %')
    axes[0].set_title(f'Missing Values — {title}')
    axes[0].legend()
    for i, v in enumerate(miss.values):
        axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8)

    # Heatmap (top 15 cols max)
    top_cols = miss.head(15).index.tolist()
    sample = df[top_cols].head(200).isnull()
    sns.heatmap(sample.T, cmap='YlOrRd', cbar=False, ax=axes[1],
                xticklabels=False, yticklabels=True)
    axes[1].set_title('Missingness pattern (first 200 rows, top 15 cols)')
    axes[1].set_xlabel('Row index')

    plt.suptitle(f'Missing Value Report: [{title}]', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Drop recommendations
    to_drop = miss[miss > HIGH_MISSING_THRESHOLD * 100]
    if not to_drop.empty:
        print(f'\n🔴 Columns to consider DROPPING (>{HIGH_MISSING_THRESHOLD*100:.0f}% missing):')
        display(to_drop.reset_index().rename(columns={'index':'column', 0:'missing_%'}))


if ENDPOINT_TO_PROFILE in dfs:
    plot_missing(dfs[ENDPOINT_TO_PROFILE], title=ENDPOINT_TO_PROFILE, threshold=0)

In [ ]:
# Missing overview cho TẤT CẢ endpoints
miss_summary = []
for ep, df in dfs.items():
    miss_pct = df.isnull().mean() * 100
    miss_summary.append({'endpoint': ep, 'rows': len(df),
                          'cols_total': len(df.columns),
                          'cols_any_null': (miss_pct > 0).sum(),
                          'cols_drop_candidate': (miss_pct > HIGH_MISSING_THRESHOLD*100).sum(),
                          'overall_null_%': round(df.isnull().values.mean() * 100, 2)})

miss_df = pd.DataFrame(miss_summary)
print('=== Missing Value Summary (All Endpoints) ===')
display(miss_df.style.background_gradient(cmap='Reds', subset=['overall_null_%', 'cols_drop_candidate']))

---
## 6️⃣ Deduplication Check

In [ ]:
KEY_MAP = {
    'sessions':       ['session_key'],
    'drivers':        ['session_key', 'driver_number'],
    'laps':           ['session_key', 'driver_number', 'lap_number'],
    'stints':         ['session_key', 'driver_number', 'stint_number'],
    'weather':        ['session_key', 'date'],
    'intervals':      ['session_key', 'driver_number', 'date'],
    'starting_grid':  ['session_key', 'driver_number'],
    'session_result': ['session_key', 'driver_number'],
    'meetings':       ['meeting_key'],
}

dup_results = []
for ep, df in dfs.items():
    keys = [k for k in KEY_MAP.get(ep, []) if k in df.columns]
    # Full row duplicates
    full_dups = df.duplicated().sum()
    # Key duplicates
    key_dups = df.duplicated(subset=keys).sum() if keys else None
    dup_results.append({'endpoint': ep, 'total_rows': len(df),
                         'full_row_dups': full_dups,
                         'full_dup_%': round(full_dups/len(df)*100, 2),
                         'key_cols': str(keys),
                         'key_dups': key_dups})

dup_df = pd.DataFrame(dup_results)
print('=== Deduplication Report ===')
display(dup_df.style.background_gradient(cmap='Oranges', subset=['full_dup_%']))

---
## 7️⃣ Datetime Feature Detection & Transformation Check

In [ ]:
import re

DATE_PATTERNS = re.compile(r'date|time|start|end|created|updated', re.IGNORECASE)

dt_report = []
for ep, df in dfs.items():
    for col in df.select_dtypes(include='object').columns:
        if DATE_PATTERNS.search(col):
            sample = df[col].dropna().head(3).tolist()
            # Try parsing
            try:
                parsed = pd.to_datetime(df[col].dropna().head(100), errors='coerce')
                parse_rate = parsed.notna().mean() * 100
            except Exception:
                parse_rate = 0
            dt_report.append({'endpoint': ep, 'column': col,
                               'current_dtype': str(df[col].dtype),
                               'parse_success_%': round(parse_rate, 1),
                               'sample': str(sample[:2]),
                               'action': '✅ Convert to datetime' if parse_rate > 80 else '⚠️ Check format'})

if dt_report:
    print('=== Datetime Column Detection ===')
    display(pd.DataFrame(dt_report).style.applymap(
        lambda v: 'background-color: #d4edda' if '✅' in str(v) else
                  'background-color: #fff3cd' if '⚠️' in str(v) else '',
        subset=['action']
    ))
else:
    print('No datetime-like columns detected in object columns')

---
## 📋 Tổng hợp: Cleaning Recommendations

In [ ]:
print('=' * 70)
print('CLEANING STRATEGY SUMMARY')
print('=' * 70)

for ep, df in dfs.items():
    print(f'\n📁 [{ep}]  {len(df):,} rows × {len(df.columns)} cols')

    # Missing
    miss = df.isnull().mean() * 100
    drop_cols = miss[miss > HIGH_MISSING_THRESHOLD * 100].index.tolist()
    impute_cols = miss[(miss > 0) & (miss <= HIGH_MISSING_THRESHOLD * 100)].index.tolist()
    if drop_cols:
        print(f'  🔴 DROP (>{HIGH_MISSING_THRESHOLD*100:.0f}% missing): {drop_cols}')
    if impute_cols:
        print(f'  🟡 IMPUTE: {impute_cols[:5]}{"..." if len(impute_cols)>5 else ""}')

    # Low variance
    lv = [c for c in df.columns if c != '_source_endpoint' and
           df[c].value_counts(normalize=True, dropna=False).iloc[0] >= LOW_VARIANCE_THRESHOLD]
    if lv:
        print(f'  ⚪ LOW VARIANCE (consider drop): {lv}')

    # Duplicates
    dups = df.duplicated().sum()
    if dups > 0:
        print(f'  🟠 DUPLICATES: {dups:,} full-row duplicates')

    # Negative numbers
    neg = {c: (df[c] < 0).sum() for c in df.select_dtypes(include='number').columns if (df[c] < 0).any()}
    if neg:
        print(f'  🔴 NEGATIVE VALUES: {neg}')

print('\n' + '=' * 70)
print('Done! Use these findings to tune src/clean_data.py 🚀')
print('=' * 70)